# get similarity convergence

In [28]:
import os
import platform

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

In [29]:
os_system = platform.system() # 맥북은 Darwin, 윈도우는 Windows

# 현재 프로젝트 폴더 위치 지정. os.getcwd()는 지금 코드 실행하는 현 위치를 출력해줍니다.
research1_dir = os.getcwd()

# data/processed 폴더 위치 지정
processed_data_dir = research1_dir + ('\\data\\processed\\' if os_system == 'Windows' else '/data/processed/')

# graph 이미지 저장할 폴더 위치 지정
graph_image_dir = research1_dir + ('\\graph\\slope' if os_system == 'Windows' else '/graph/slope')

In [30]:
# 폴더 없으면 생성
os.makedirs(graph_image_dir, exist_ok=True)

In [31]:
# 유사도, 유사도평균 coherence값을 저장한 테이블 읽어오기
tbl_data = pd.read_csv(processed_data_dir + 'similarity_coherence.csv', index_col=0, keep_default_na=False)
tbl_data[0:3]

,similarity_abuse1_money,similarity_abuse2_money,similarity_abuse3_money,similarity_abuse4_money,similarity_abuse5_money,similarity_abuse6_money,similarity_abuse7_money,similarity_abuse8_money,similarity_abuse9_money,similarity_abuse10_money,...,coherence_abuse_relationships,coherence_tear_relationships,coherence_mirror_relationships,coherence_family_relationships,coherence_relationships,coherence_abuse_family,coherence_tear_family,coherence_mirror_family,coherence_family_family,coherence_family
subject,,,,,,,,,,,,,,,,,,,,,
1,0.11029749613397022,0.14175797893698405,0.09668021562052198,0.22584298643295497,0.14024075254713653,0.24353777244973585,0.05485440967803956,0.17019819717279439,0.24451821664000506,0.2878031719073627,...,0.11065202713638661,0.10437839139747804,0.03029209833241308,0.11159175646637291,0.08922856833316266,0.17904907556024252,0.20934783739354912,0.11293917958196284,0.12487956321975098,0.15655391393887638
2,0.1440517533857011,0.1440517533857011,0.09284857489315534,0.08648248097104261,0.06925226964638698,0.13071754093043397,0.0494621453782329,0.6077935143730266,0.08810924674340304,0.17968282304011507,...,0.052968230049086,0.06098017328939255,0.07189595857478095,0.05014490834752351,0.05899731756519576,0.10678515799590696,0.06944599572042942,0.1554019398539225,0.10252325282708308,0.10853908659933549
3,0.1769635821237211,0.15549655562743292,-0.045841510315637324,0.12451748211442537,0.10273576991616806,0.22273224762158228,0.08812668184381556,0.059070508911501785,-0.025725836170088723,0.029017753550611758,...,0.04626450524398483,0.07242303085693248,0.06651371143000759,0.061945703535651195,0.06178673776664402,0.10539788824477483,0.13093511750786943,0.12230413679937391,0.1341830711968474,0.12320505343721638


In [32]:
def get_smoothed_similarity_func(similarity_df: pd.DataFrame, i_subject: int, topic: str):
    # trial 번호와 해당 trial에 대한 similarity 값을 배열로 변환
    trial_numbers = np.array(list(range(1, 41)))

    # 특정 피험자의 유사도 점수들 받아오기
    similarity_values = similarity_df.iloc[i_subject].tolist()
    similarity_values = np.array([float(value) if value != '' else 0.0 for value in similarity_values])

    # 데이터를 보간하는 함수 생성
    interpolation_function = interp1d(trial_numbers, similarity_values, kind='quadratic')

    # 정수값에 대한 데이터 추출
    integer_trial_numbers = np.arange(1, 41)
    integer_similarity_values = interpolation_function(integer_trial_numbers)

    # 부드러운 곡선을 위해 trial 번호를 더 자세히 나누기
    fine_trial_numbers = np.linspace(1, 40, 400)
    smoothed_similarity_values = interpolation_function(fine_trial_numbers)

    # 1차 함수 (선형 회귀)를 생성하여 예측값 얻기
    z = np.polyfit(integer_trial_numbers, integer_similarity_values, 1)
    p = np.poly1d(z)
    predicted_values = p(integer_trial_numbers)

    # 기울기와 절편 구하기
    slope = z[0]
    intercept = z[1]

    # 그래프 저장할 위치
    graph_image_path = (f'\\graph\\{topic}\\Subject_{i_subject}_similarity_curve.png' if os_system == 'Windows' else f'{graph_image_dir}/{topic}/Subject_{i_subject}_similarity_curve.png')
    # 폴더 없으면 생성
    os.makedirs(f'\\graph\\{topic}' if os_system == 'Windows' else f'{graph_image_dir}/{topic}', exist_ok=True)
    
    # 그래프 생성
    plt.figure(figsize=(10, 5))
    plt.plot(integer_trial_numbers, predicted_values, label='Linear Regression', color='g', linestyle='-')
    plt.plot(fine_trial_numbers, smoothed_similarity_values, label='Smoothed similarities', color='r')
    plt.scatter(integer_trial_numbers, integer_similarity_values, label='Real similarity Data', marker='o', color='b')
    plt.xlabel('Trial')
    plt.ylabel('Similarity Value')
    plt.title(f'Subject {i_subject}: Smoothed Similarity Curve')
    plt.legend()
    plt.grid(True)
    plt.savefig(graph_image_path)
    # plt.show()

    # 그래프 표시하지 않음
    plt.close()

    return predicted_values, (slope, intercept)


In [33]:
def add_convergence_values_to_df(start_index: int,end_index: int, dataframe: pd.DataFrame, seed_word: str, target_word: str):
    similarity_seed_target = tbl_data.iloc[:, start_index:end_index]
    slopes = []
    intercepts = []

    for i_subject in range(len(dataframe)):
        predicted_values, (slope, intercept) = get_smoothed_similarity_func(similarity_df = similarity_seed_target,
                                                                            i_subject = i_subject,
                                                                            topic = f'{seed_word}_{target_word}')
        slopes.append(slope)
        intercepts.append(intercept)
        
    # 모든 피험자의 convergence값을 얻은 후,
    tbl_data[f'convergence_slope_{seed_word}_{target_word}'] = slopes
    tbl_data[f'convergence_intercept_{seed_word}_{target_word}'] = intercepts

### target: money

In [34]:
tbl_data.iloc[:, 0:40]

,similarity_abuse1_money,similarity_abuse2_money,similarity_abuse3_money,similarity_abuse4_money,similarity_abuse5_money,similarity_abuse6_money,similarity_abuse7_money,similarity_abuse8_money,similarity_abuse9_money,similarity_abuse10_money,...,similarity_abuse31_money,similarity_abuse32_money,similarity_abuse33_money,similarity_abuse34_money,similarity_abuse35_money,similarity_abuse36_money,similarity_abuse37_money,similarity_abuse38_money,similarity_abuse39_money,similarity_abuse40_money
subject,,,,,,,,,,,,,,,,,,,,,
1,0.11029749613397022,0.14175797893698405,0.09668021562052198,0.22584298643295497,0.14024075254713653,0.24353777244973585,0.05485440967803956,0.17019819717279439,0.24451821664000506,0.2878031719073627,...,0.10877018298844798,0.22243986948194583,0.12222951765128254,0.06929840868420944,0.11156765656925827,,0.12476383471506458,,0.022682271608305493,0.0420862724906883
2,0.1440517533857011,0.1440517533857011,0.09284857489315534,0.08648248097104261,0.06925226964638698,0.13071754093043397,0.0494621453782329,0.6077935143730266,0.08810924674340304,0.17968282304011507,...,0.07854960615308804,0.07854960615308804,0.0829348412994757,0.07200924623613358,0.029360112350631407,0.06925226964638698,0.1425521559513402,,0.137540146163867,0.15693280833305523
3,0.1769635821237211,0.15549655562743292,-0.045841510315637324,0.12451748211442537,0.10273576991616806,0.22273224762158228,0.08812668184381556,0.059070508911501785,-0.025725836170088723,0.029017753550611758,...,0.03167607907748382,0.04769051784805001,0.11699662706139002,0.034094145960097744,0.049163074379439786,-0.011169995659561005,0.09870182012218498,0.098586616519454,-0.056156142811918164,0.14436561789556002
4,0.13005050781305705,0.1769635821237211,0.12605638957538956,0.17388049168511044,0.2878031719073627,1.0,0.18245410970361275,0.018326025272020985,0.057967303528177916,0.061645185300340244,...,-0.04646466929904158,0.0754344411387099,0.001550678330954347,0.1307101931605057,0.05149245756351417,0.1440517533857011,0.11555688297044953,0.0952014365308348,0.0742625004414672,-0.026741677682260656
5,0.11029749613397022,0.08605278032346253,0.1071332866949215,0.18431413072556746,0.1902941377949171,0.11286439560208184,0.1312229409996144,0.14188570732189043,0.0684194697904752,0.0516831593724576,...,-0.012871952519054153,0.22273224762158228,0.19693554754977782,0.16507237090601645,0.02472789636898698,0.08362827559724972,0.07816789075193309,0.02674354323238337,-0.02816560385168909,0.010198599422724608
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
346,0.0939911336376722,0.12222951765128254,0.09324129642525125,0.13794067981988245,0.10702293525529705,0.045827563476325994,0.08201143947145073,0.11156765656925827,0.06597401568202566,-0.01301020894824334,...,,0.08414064065124927,0.1313294600033692,0.059070508911501785,0.06787144525839883,0.07121992977254799,0.1873647171961762,0.05698373520908828,0.15784412209248966,0.0011619385911016966
347,0.12854263382364528,0.07629267147455587,0.05270128319521383,0.12222951765128254,0.19277407059389762,0.044095263034415244,0.09923448031308091,-0.07310578178941762,0.1637730116408711,0.013331097204634434,...,0.0008760442816808656,0.05829992668802009,0.09751352271841385,0.036951260626608584,-0.05331028476602895,0.02334814570040622,-0.05277746126692051,0.009584748439179447,0.07367406033223789,0.06466130954707838
348,0.09763002044425118,0.08852797523105349,0.019751086055118328,0.07284537809936509,0.10985657593682707,0.02353775839681138,0.07336701976615623,0.08936850754895065,0.10985657593682707,0.15614464721997523,...,,-0.0005472494934339878,0.04865088112147298,0.21350883304230317,,0.02773686461802438,,0.025169565757158674,0.06676248606562041,0.001110836284313521


In [35]:
add_convergence_values_to_df(start_index=0,
                             end_index=40,
                             dataframe=tbl_data,
                             seed_word='abuse',
                             target_word='money')
add_convergence_values_to_df(start_index=40,
                             end_index=80,
                             dataframe=tbl_data,
                             seed_word='tear',
                             target_word='money')
add_convergence_values_to_df(start_index=80,
                             end_index=120,
                             dataframe=tbl_data,
                             seed_word='mirror',
                             target_word='money')
add_convergence_values_to_df(start_index=120,
                             end_index=160,
                             dataframe=tbl_data,
                             seed_word='family',
                             target_word='money')

### target: friend

In [36]:
add_convergence_values_to_df(start_index=160,
                             end_index=200,
                             dataframe=tbl_data,
                             seed_word='abuse',
                             target_word='friend')
add_convergence_values_to_df(start_index=200,
                             end_index=240,
                             dataframe=tbl_data,
                             seed_word='tear',
                             target_word='friend')
add_convergence_values_to_df(start_index=240,
                             end_index=280,
                             dataframe=tbl_data,
                             seed_word='mirror',
                             target_word='friend')
add_convergence_values_to_df(start_index=280,
                             end_index=320,
                             dataframe=tbl_data,
                             seed_word='family',
                             target_word='friend')

### target: relationships

In [37]:
add_convergence_values_to_df(start_index=320,
                             end_index=360,
                             dataframe=tbl_data,
                             seed_word='abuse',
                             target_word='relationships')
add_convergence_values_to_df(start_index=360,
                             end_index=400,
                             dataframe=tbl_data,
                             seed_word='tear',
                             target_word='relationships')
add_convergence_values_to_df(start_index=400,
                             end_index=440,
                             dataframe=tbl_data,
                             seed_word='mirror',
                             target_word='relationships')
add_convergence_values_to_df(start_index=440,
                             end_index=480,
                             dataframe=tbl_data,
                             seed_word='family',
                             target_word='relationships')

### target: family

In [38]:
add_convergence_values_to_df(start_index=480,
                             end_index=520,
                             dataframe=tbl_data,
                             seed_word='abuse',
                             target_word='family')
add_convergence_values_to_df(start_index=520,
                             end_index=560,
                             dataframe=tbl_data,
                             seed_word='tear',
                             target_word='family')
add_convergence_values_to_df(start_index=560,
                             end_index=600,
                             dataframe=tbl_data,
                             seed_word='mirror',
                             target_word='family')
add_convergence_values_to_df(start_index=600,
                             end_index=640,
                             dataframe=tbl_data,
                             seed_word='family',
                             target_word='family')

## save csv

In [39]:
tbl_data[0:3]

,similarity_abuse1_money,similarity_abuse2_money,similarity_abuse3_money,similarity_abuse4_money,similarity_abuse5_money,similarity_abuse6_money,similarity_abuse7_money,similarity_abuse8_money,similarity_abuse9_money,similarity_abuse10_money,...,convergence_slope_family_relationships,convergence_intercept_family_relationships,convergence_slope_abuse_family,convergence_intercept_abuse_family,convergence_slope_tear_family,convergence_intercept_tear_family,convergence_slope_mirror_family,convergence_intercept_mirror_family,convergence_slope_family_family,convergence_intercept_family_family
subject,,,,,,,,,,,,,,,,,,,,,
1,0.11029749613397022,0.14175797893698405,0.09668021562052198,0.22584298643295497,0.14024075254713653,0.24353777244973585,0.05485440967803956,0.17019819717279439,0.24451821664000506,0.2878031719073627,...,0.001196,0.087068,-0.004465,0.257158,0.000234,0.204547,0.003148,0.048414,-0.000748,0.140217
2,0.1440517533857011,0.1440517533857011,0.09284857489315534,0.08648248097104261,0.06925226964638698,0.13071754093043397,0.0494621453782329,0.6077935143730266,0.08810924674340304,0.17968282304011507,...,-0.000563,0.059169,-0.002800,0.158849,-0.001593,0.100376,-0.000017,0.151856,-0.002405,0.146696
3,0.1769635821237211,0.15549655562743292,-0.045841510315637324,0.12451748211442537,0.10273576991616806,0.22273224762158228,0.08812668184381556,0.059070508911501785,-0.025725836170088723,0.029017753550611758,...,-0.000106,0.064118,-0.003434,0.175805,0.001286,0.098026,-0.000334,0.129158,-0.000194,0.138164


In [40]:
# 단어 있는 버전 csv 저장
tbl_data.to_csv(processed_data_dir + 'convergence.csv')
